In [1]:
library(reshape2)
library(Seurat)
library(dplyr)
library(scales)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


corrplot 0.92 loaded



── SCpubr 2.0.2.9000 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

ℹ Have a look at extensive tutorials in SCpubr's book.

✔ If you use SCpubr in your research, please cite it accordingly.

★ If the package is useful to you, consider leaving a Star in the GitHub repository.

! Keep track of the package updates on Twitter (@Enblacar) or in the Official NEWS website.

♥ Happy plotting!



── Tips! ──

! Check missing dependencies with: SC

Data only avalibale as processed data

## Smart-seq2

In [18]:
expr <- read.table("/projects/0/einf2548/cruiz/dmg/public_data/LaBelle2025/GSE227983_immune_ss2_tpm.txt.gz", header = TRUE, row.names = 1, sep = "\t", check.names = FALSE)
meta <- read.table("/projects/0/einf2548/cruiz/dmg/public_data/LaBelle2025/GSE227983_immune_ss2_meta.txt.gz", header = TRUE, row.names = 1, sep = "\t")

In [22]:
expr <- expr[, rownames(meta)]
labelle <- CreateSeuratObject(counts = expr, meta.data = meta)
labelle <- NormalizeData(labelle, normalization.method = "LogNormalize", scale.factor = 10000) %>% FindVariableFeatures() %>% ScaleData()
labelle

Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”
Normalizing layer: counts

Finding variable features for layer counts

Centering and scaling data matrix



An object of class Seurat 
19357 features across 4158 samples within 1 assay 
Active assay: RNA (19357 features, 2000 variable features)
 3 layers present: counts, data, scale.data

In [25]:
# Filter donors to include only DMG and DG-H3WT
subtype_values <- c("Hemispheric-HistoneWT", "Midline-H3K27M", "Midline-HistoneWT")
# only include myeloid cells
labelle <- subset(
  labelle,
  subset = Subtype %in% subtype_values & detailed_annot == "Myeloid"
)
labelle

Warning message:
“Removing 171 cells missing data for vars requested”


An object of class Seurat 
19357 features across 676 samples within 1 assay 
Active assay: RNA (19357 features, 2000 variable features)
 3 layers present: counts, data, scale.data

In [ ]:
saveRDS(labelle, 'labelle_smartseq2.rds')

## 10X genomics

In [2]:
expr <- read.table("/projects/0/einf2548/cruiz/dmg/public_data/LaBelle2025/GSE227983_immune_tenx_tpm.txt.gz", header = TRUE, row.names = 1, sep = "\t", check.names = FALSE)
meta <- read.table("/projects/0/einf2548/cruiz/dmg/public_data/LaBelle2025/GSE227983_immune_tenx_meta.txt.gz", header = TRUE, row.names = 1, sep = "\t")

In [3]:
labelle <- CreateSeuratObject(counts = expr, meta.data = meta)
labelle <- NormalizeData(labelle, normalization.method = "LogNormalize", scale.factor = 10000) %>% FindVariableFeatures() %>% ScaleData()
labelle

Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”
Normalizing layer: counts

Finding variable features for layer counts

Centering and scaling data matrix



An object of class Seurat 
19357 features across 25050 samples within 1 assay 
Active assay: RNA (19357 features, 2000 variable features)
 3 layers present: counts, data, scale.data

In [5]:
table(meta$Subtype)


          BT1857_TenX                BT2062           BT2080_TenX 
                  767                  5189                  6358 
               BT2082                BT2083 Hemispheric-HistoneWT 
                 6398                  5571                   767 

In [13]:
# Based on data in the paper and supplementary tables, these are the donors that match DMG/DG-H3WT
labelle <- subset(
  labelle,
  subset = Subtype %in% c('BT1857_TenX', 'BT2080_TenX', 'Hemispheric-HistoneWT', 'BT2062') & broad_annot == "Myeloid"
)
labelle

Warning message:
“Removing 767 cells missing data for vars requested”


An object of class Seurat 
19357 features across 8631 samples within 1 assay 
Active assay: RNA (19357 features, 2000 variable features)
 3 layers present: counts, data, scale.data

In [ ]:
saveRDS(labelle, 'labelle_10x.rds')